<a href="https://colab.research.google.com/github/jayaprakash-kolla/MTechThesis-EEG2TexT/blob/main/preprocessing_for_eeg_data.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# ---------- code for checking any reference channels are present at session level or at dataset level

import json
import os

# Check session-level JSON
session_json = "/content/drive/MyDrive/ds004395/sub-LTP324/ses-8/eeg/sub-LTP324_ses-8_task-ltpFR2_eeg.json"
if os.path.exists(session_json):
    with open(session_json) as f:
        print("Session JSON EEGReference:", json.load(f).get('EEGReference'))

# Check dataset-level JSON (this might be the culprit!)
dataset_json = "/content/drive/MyDrive/ds004395/task-ltpFR2_eeg.json"
if os.path.exists(dataset_json):
    with open(dataset_json) as f:
        print("Dataset JSON EEGReference:", json.load(f).get('EEGReference'))
else:
    print("No dataset-level JSON found")

# Also check the channels.tsv
channels_tsv = "/content/drive/MyDrive/ds004395/sub-LTP324/ses-8/eeg/sub-LTP324_ses-8_task-ltpFR2_channels.tsv"
if os.path.exists(channels_tsv):
    import pandas as pd
    df = pd.read_csv(channels_tsv, sep='\t')
    print("\nChannels TSV columns:", df.columns.tolist())
    if 'reference' in df.columns:
        print("References in TSV:", df['reference'].unique())

Session JSON EEGReference: E129
No dataset-level JSON found

Channels TSV columns: ['name', 'type', 'units', 'low_cutoff', 'high_cutoff', 'description', 'sampling_frequency', 'status', 'status_description']


In [11]:
# code for loading the EEG data directly without using the bids module

from mne.io import read_raw_edf

bids_root = r"/content/drive/MyDrive/ds004395" # Update this path to where your dataset is located in Colab
subject_id = 'LTP324'
task = 'ltpFR2'
session_id = '8'

# Loading the EEG data directly and bypassing the BIDS completly.
edf_file = f"{bids_root}/sub-{subject_id}/ses-{session_id}/eeg/sub-{subject_id}_ses-{session_id}_task-{task}_eeg.edf"
raw = read_raw_edf(edf_file, preload=True, verbose=False)

# Set your reference
raw.set_eeg_reference(ref_channels=['E129'])  # or 'average' or whatever you need

print(f"Loaded raw data: {raw.info['nchan']} channels, {raw.info['sfreq']} Hz.")

In [ ]:
# Clear the path and reload the bids
import importlib
importlib.reload(mne_bids)

In [7]:
# code for checking what channels are present in the EEG data.

from mne.io import read_raw_edf

edf_file = f"/content/drive/MyDrive/ds004395/sub-LTP324/ses-8/eeg/sub-LTP324_ses-8_task-ltpFR2_eeg.edf"
raw_direct = read_raw_edf(edf_file, preload=False)
print("Actual channels in EDF:", raw_direct.ch_names)
print("Does E129 exist?", 'E129' in raw_direct.ch_names)
print("Does Cz exist?", 'Cz' in raw_direct.ch_names)

Extracting EDF parameters from /content/drive/MyDrive/ds004395/sub-LTP324/ses-8/eeg/sub-LTP324_ses-8_task-ltpFR2_eeg.edf...
Setting channel info structure...
Creating raw.info structure...
Actual channels in EDF: ['E1', 'E2', 'E3', 'E4', 'E5', 'E6', 'E7', 'E8', 'E9', 'E10', 'E11', 'E12', 'E13', 'E14', 'E15', 'E16', 'E17', 'E18', 'E19', 'E20', 'E21', 'E22', 'E23', 'E24', 'E25', 'E26', 'E27', 'E28', 'E29', 'E30', 'E31', 'E32', 'E33', 'E34', 'E35', 'E36', 'E37', 'E38', 'E39', 'E40', 'E41', 'E42', 'E43', 'E44', 'E45', 'E46', 'E47', 'E48', 'E49', 'E50', 'E51', 'E52', 'E53', 'E54', 'E55', 'E56', 'E57', 'E58', 'E59', 'E60', 'E61', 'E62', 'E63', 'E64', 'E65', 'E66', 'E67', 'E68', 'E69', 'E70', 'E71', 'E72', 'E73', 'E74', 'E75', 'E76', 'E77', 'E78', 'E79', 'E80', 'E81', 'E82', 'E83', 'E84', 'E85', 'E86', 'E87', 'E88', 'E89', 'E90', 'E91', 'E92', 'E93', 'E94', 'E95', 'E96', 'E97', 'E98', 'E99', 'E100', 'E101', 'E102', 'E103', 'E104', 'E105', 'E106', 'E107', 'E108', 'E109', 'E110', 'E111', 'E112'

In [8]:
!pip install mne
!pip install mne-bids
!pip install tensorflow

In [4]:
import os
import mne
from mne.io import read_raw_edf
import mne_bids
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import RobustScaler
# Note: ASR import has been removed as it's not in your function

In [10]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [5]:
# --- HELPER FUNCTIONS ---

def preprocess_raw(raw):
    """
    Applies optimized preprocessing:
    1. Sets EOG types
    2. High-pass filters (for ASR)
    3. Applies ASR
    4. Low-pass filters
    5. Re-references to the 129th channel
    6. Downsamples to 250 Hz
    """

    print("---------------entered the preprocess_raw method---------------")

    # CRITICAL: Ensure data is loaded into memory
    raw.load_data()

    # 1. Handle EOG Channels (Based on previous warnings)
    eog_channels = ['E8', 'E26', 'E126', 'E127']
    # Filter EOG channels that are actually present in raw.ch_names
    eog_channels_present = [ch for ch in eog_channels if ch in raw.ch_names]
    if eog_channels_present:
        raw.set_channel_types({ch: 'eog' for ch in eog_channels_present})
        print(f"    ... Set EOG types for: {eog_channels_present}")
    else:
        print(f"    ... No specified EOG channels found in raw data.")

    # 2. High-pass filter (required for ASR)
    raw.filter(l_freq=0.5, h_freq=None, fir_design='firwin', verbose=False)

    # 4. Low-pass filter (after ASR)
    raw.filter(l_freq=None, h_freq=40.0, fir_design='firwin', verbose=False)

    # 5. Re-reference to the 129th channel or 'Cz'
    # This is the section causing the 'Cz' is not in list error.
    # Let's make it more robust.

    # First, list all channels for debugging
    print(f"    ... All channels before re-referencing: {raw.ch_names}")

    ref_channel_name = None
    if 'Cz' in raw.ch_names:
        ref_channel_name = 'Cz'
        print(f"    ... Found 'Cz' in channel list. Using 'Cz' for re-referencing.")
    elif len(raw.ch_names) >= 129:
        # If 'Cz' is not explicitly named, but the comment refers to 129th channel,
        # let's try using the 129th channel (index 128) if it exists.
        ref_channel_name = raw.ch_names[128]
        print(f"    ... 'Cz' not found. Using 129th channel (index 128): {ref_channel_name} for re-referencing.")
    elif len(raw.ch_names) > 0:
        # Fallback to the last channel if there are fewer than 129 channels and 'Cz' is not found. # Fallback to the last channel if there are fewer than 129 channels and 'Cz' is not found.
        ref_channel_name = raw.ch_names[-1]
        print(f"    ... Less than 129 channels and 'Cz' not found. Using last channel: {ref_channel_name} for re-referencing.")
    else:
        raise ValueError("No channels available to determine a reference for re-referencing.")

    if ref_channel_name is None:
        raise ValueError("Could not determine a reference channel for re-referencing.")

    # Ensure the chosen reference channel is in the current list of channels,
    # just in case something subtle removed it between checks.
    if ref_channel_name not in raw.ch_names:
        raise ValueError(f"Chosen reference channel '{ref_channel_name}' not found in current raw.ch_names list: {raw.ch_names}")

    # Proceed with re-referencing
    print(f"    ... Re-referencing to channel: {ref_channel_name}")
    raw.set_eeg_reference(ref_channels=[ref_channel_name], verbose=False)


    # 6. Downsample to 250 Hz
    print(f"    ... Downsampling from {raw.info['sfreq']} Hz to 250 Hz")
    raw.resample(sfreq=250, verbose=False)

    print(f"--------------returning the raw data object and exiting the preprocess_raw function")

    return raw

In [6]:
def label_and_epoch_data(raw, session_id):
    """
    Performs cross-event linkage for RECALL only and creates X and Y arrays.
    """
    print("---------------entered the label_and_epoch_data method---------------")

    # 1. Get the full annotations DataFrame
    full_metadata = raw.annotations.to_data_frame()
    full_metadata['item_name'] = full_metadata['item_name'].astype(str)
    full_metadata['session'] = session_id

    # 2. Create Lookup Table (Recall Only)
    recalled_words_set = set(full_metadata[full_metadata['description'] == 'REC_WORD']['item_name'].unique())
    recalled_words_set.discard('nan'); recalled_words_set.discard('n/a')

    # 3. Create Master Annotation Table for WORD Events
    word_annots_df = full_metadata[full_metadata['description'] == 'WORD'].reset_index(drop=True)

    # Determine Final Outcomes for Each WORD Event
    word_annots_df['is_recalled'] = word_annots_df['item_name'].apply(
        lambda x: 1 if x in recalled_words_set else 0
    )

    # 4. Get the MNE events array (using ALL WORD events)
    word_event_id = {'WORD': 1}
    events, _ = mne.events_from_annotations(raw, event_id=word_event_id, verbose=False)

    # 5. FINAL X/Y STRUCTURING
    # We use word_annots_df (unfiltered) and events (unfiltered)

    # Y Data: Create final behavioral table
    Y_df = word_annots_df[[
        'onset', 'item_name', 'session', 'is_recalled'
    ]].copy()

    # X Data: Epoch the data
    epochs = mne.Epochs(
        raw,
        events, # <-- Use the full events array
        event_id=word_event_id,
        tmin=-0.2, tmax=1.0,
        preload=True,
        baseline=(-0.2, 0),
        verbose=False
    )
    X = epochs.get_data()

    # We must ensure the events array and the annots_df align
    # This check is now more complex, but for simple 'WORD' extraction, it should hold
    if len(X) != len(Y_df):
        raise RuntimeError(f"X and Y arrays are out of sync! X={len(X)}, Y={len(Y_df)}")

    print(f"--------------returning the labelled data and exiting the label_and_epoch_data function")

    return X, Y_df

In [13]:

'''
  Here we are not looping over all the sessions. We are trying to load the data for only one session.

'''

# --- CONFIGURATION ---
bids_root = r"/content/drive/MyDrive/ds004395" # Update this path to where your dataset is located in Colab
subject_id = 'LTP324'
task = 'ltpFR2'
session_id = '8'

all_X_data = []
all_Y_data = []

# --- MAIN LOOP ---

print(f"Starting pipeline for Subject: {subject_id}")
#for session_id in all_sessions: ---- removing this commentr can cause the intendation error.
print(f"\n--- Processing Session {session_id} ---")

try:

    # 1. Load Data
    bids_path = mne_bids.BIDSPath(
        subject=subject_id, session=session_id, datatype='eeg',
        task=task, root=bids_root
    )
    print(f"looking for path :: {bids_path}")

    raw = mne_bids.read_raw_bids(bids_path=bids_path, verbose=True, extra_params={'preload': False}, on_ch_mismatch='reorder')
    print(f"    Loaded raw data: {raw.info['nchan']} channels, {raw.info['sfreq']} Hz.")
    print(f"    Channels found in raw data after loading: {raw.ch_names}") # <-- Added this line for debugging

    # 2. Preprocess (Using your provided function)
    raw = preprocess_raw(raw)

    # 3. Epoch and Label
    X_session, Y_session = label_and_epoch_data(raw, session_id)

    # 4. Store Results
    all_X_data.append(X_session)
    all_Y_data.append(Y_session)

    print(f"    Session {session_id} successful. Extracted {len(X_session)} epochs.")

except FileNotFoundError:
    print(f"❌ File not found for Session {session_id}. Skipping.")
except Exception as e:
    print(f"❌ An error occurred in Session {session_id}: {e}")



Starting pipeline for Subject: LTP324

--- Processing Session 8 ---
looking for path :: /content/drive/MyDrive/ds004395/sub-LTP324/ses-8/eeg/sub-LTP324_ses-8_task-ltpFR2_eeg.edf
Extracting EDF parameters from /content/drive/MyDrive/ds004395/sub-LTP324/ses-8/eeg/sub-LTP324_ses-8_task-ltpFR2_eeg.edf...
Setting channel info structure...
Creating raw.info structure...
Reading events from /content/drive/MyDrive/ds004395/sub-LTP324/ses-8/eeg/sub-LTP324_ses-8_task-ltpFR2_events.tsv.
Reading channel info from /content/drive/MyDrive/ds004395/sub-LTP324/ses-8/eeg/sub-LTP324_ses-8_task-ltpFR2_channels.tsv.
Reading electrode coords from /content/drive/MyDrive/ds004395/sub-LTP324/ses-8/eeg/sub-LTP324_ses-8_space-CapTrak_electrodes.tsv.
Not fully anonymizing info - keeping his_id, sex, and hand info
    Loaded raw data: 129 channels, 500.0 Hz.
    Channels found in raw data after loading: ['E1', 'E2', 'E3', 'E4', 'E5', 'E6', 'E7', 'E8', 'E9', 'E10', 'E11', 'E12', 'E13', 'E14', 'E15', 'E16', 'E17', '

/tmp/ipython-input-3283060137.py:30: RuntimeWarning: There are channels without locations (n/a) that are not marked as bad: ['E8', 'E25', 'E126', 'E127']
  raw = mne_bids.read_raw_bids(bids_path=bids_path, verbose=True, extra_params={'preload': False}, on_ch_mismatch='reorder')
/tmp/ipython-input-3283060137.py:30: RuntimeWarning: DigMontage is only a subset of info. There is 1 channel position not present in the DigMontage. The channel missing from the montage is:

['E129'].

Consider using inst.rename_channels to match the montage nomenclature, or inst.set_channel_types if this is not an EEG channel, or use the on_missing parameter if the channel position is allowed to be unknown in your analyses.
  raw = mne_bids.read_raw_bids(bids_path=bids_path, verbose=True, extra_params={'preload': False}, on_ch_mismatch='reorder')
/tmp/ipython-input-3283060137.py:30: RuntimeWarning: Not setting positions of 4 eog channels found in montage:
['E8', 'E25', 'E126', 'E127']
Consider setting the chann

    ... Set EOG types for: ['E8', 'E26', 'E126', 'E127']
    ... All channels before re-referencing: ['E1', 'E2', 'E3', 'E4', 'E5', 'E6', 'E7', 'E8', 'E9', 'E10', 'E11', 'E12', 'E13', 'E14', 'E15', 'E16', 'E17', 'E18', 'E19', 'E20', 'E21', 'E22', 'E23', 'E24', 'E25', 'E26', 'E27', 'E28', 'E29', 'E30', 'E31', 'E32', 'E33', 'E34', 'E35', 'E36', 'E37', 'E38', 'E39', 'E40', 'E41', 'E42', 'E43', 'E44', 'E45', 'E46', 'E47', 'E48', 'E49', 'E50', 'E51', 'E52', 'E53', 'E54', 'E55', 'E56', 'E57', 'E58', 'E59', 'E60', 'E61', 'E62', 'E63', 'E64', 'E65', 'E66', 'E67', 'E68', 'E69', 'E70', 'E71', 'E72', 'E73', 'E74', 'E75', 'E76', 'E77', 'E78', 'E79', 'E80', 'E81', 'E82', 'E83', 'E84', 'E85', 'E86', 'E87', 'E88', 'E89', 'E90', 'E91', 'E92', 'E93', 'E94', 'E95', 'E96', 'E97', 'E98', 'E99', 'E100', 'E101', 'E102', 'E103', 'E104', 'E105', 'E106', 'E107', 'E108', 'E109', 'E110', 'E111', 'E112', 'E113', 'E114', 'E115', 'E116', 'E117', 'E118', 'E119', 'E120', 'E121', 'E122', 'E123', 'E124', 'E125', 'E126'

In [12]:
# --- 1. FINAL AGGREGATION ---

if all_X_data:
    X_combined = np.concatenate(all_X_data, axis=0)
    Y_combined = pd.concat(all_Y_data, ignore_index=True)

    print("\n==============================================")
    print("✅ Pipeline Complete.")
    print(f"Total Combined Trials (X): {X_combined.shape[0]}")
    print(f"Final Data Shape: (Trials, Channels, Timepoints) = {X_combined.shape}")
else:
    print("\nPipeline finished, but no data was processed.")
    X_combined = None
    Y_combined = None

# --- 2. CLASSIFIER PREPARATION (SCALING) ---

if X_combined is not None:

    # 2.1. Define Target Label
    target_label = 'is_recalled'  # <--- CORRECTED
    y = Y_combined[target_label].values

    # 2.2. Split into Training and Testing sets
    X_train, X_test, y_train, y_test = train_test_split(
        X_combined,
        y,
        test_size=0.2,
        stratify=y, # Still stratify, as recall will also be imbalanced
        random_state=42
    )

    # 2.3. Apply RobustScaler
    scaler = RobustScaler()

    n_trials_train, n_chans, n_times = X_train.shape
    X_train_reshaped = X_train.reshape(n_trials_train * n_chans, n_times)

    print("\nFitting RobustScaler on training data...")
    scaler.fit(X_train_reshaped)

    # Transform training data
    X_train_scaled_reshaped = scaler.transform(X_train_reshaped)
    X_train_scaled = X_train_scaled_reshaped.reshape(n_trials_train, n_chans, n_times)

    # Transform test data
    n_trials_test = X_test.shape[0]
    X_test_reshaped = X_test.reshape(n_trials_test * n_chans, n_times)
    X_test_scaled_reshaped = scaler.transform(X_test_reshaped)
    X_test_scaled = X_test_scaled_reshaped.reshape(n_trials_test, n_chans, n_times)

    print("✅ Data standardized and ready for classification (Task: Recall SME).")
    print(f"X_train_scaled shape: {X_train_scaled.shape}")
    print(f"X_test_scaled shape: {X_test_scaled.shape}")



✅ Pipeline Complete.
Total Combined Trials (X): 576
Final Data Shape: (Trials, Channels, Timepoints) = (576, 129, 301)

Fitting RobustScaler on training data...
✅ Data standardized and ready for classification (Task: Recall SME).
X_train_scaled shape: (460, 129, 301)
X_test_scaled shape: (116, 129, 301)


In [ ]:

'''
# this code is giving the "Cz not found in list error" and using loop to iterate over all the channels.



# --- CONFIGURATION ---
bids_root = r"/content/drive/MyDrive/ds004395" # Update this path to where your dataset is located in Colab
subject_id = 'LTP324'
task = 'ltpFR2'

all_sessions = ['8'] # Add all session IDs here
#all_sessions = ['1', '2']
# Initialize lists


all_X_data = []
all_Y_data = []


def label_and_epoch_data(raw, session_id):
    """
    Performs cross-event linkage for RECALL only and creates X and Y arrays.
    """

    # 1. Get the full annotations DataFrame
    full_metadata = raw.annotations.to_data_frame()
    full_metadata['item_name'] = full_metadata['item_name'].astype(str)
    full_metadata['session'] = session_id

    # 2. Create Lookup Table (Recall Only)
    recalled_words_set = set(full_metadata[full_metadata['description'] == 'REC_WORD']['item_name'].unique())
    recalled_words_set.discard('nan'); recalled_words_set.discard('n/a')

    # 3. Create Master Annotation Table for WORD Events
    word_annots_df = full_metadata[full_metadata['description'] == 'WORD'].reset_index(drop=True)

    # Determine Final Outcomes for Each WORD Event
    word_annots_df['is_recalled'] = word_annots_df['item_name'].apply(
        lambda x: 1 if x in recalled_words_set else 0
    )

    # 4. Get the MNE events array (using ALL WORD events)
    word_event_id = {'WORD': 1}
    events, _ = mne.events_from_annotations(raw, event_id=word_event_id, verbose=False)

    # 5. FINAL X/Y STRUCTURING
    # We use word_annots_df (unfiltered) and events (unfiltered)

    # Y Data: Create final behavioral table
    Y_df = word_annots_df[[
        'onset', 'item_name', 'session', 'is_recalled'
    ]].copy()

    # X Data: Epoch the data
    epochs = mne.Epochs(
        raw,
        events, # <-- Use the full events array
        event_id=word_event_id,
        tmin=-0.2, tmax=1.0,
        preload=True,
        baseline=(-0.2, 0),
        verbose=False
    )
    X = epochs.get_data()

    # We must ensure the events array and the annots_df align
    # This check is now more complex, but for simple 'WORD' extraction, it should hold
    if len(X) != len(Y_df):
        raise RuntimeError(f"X and Y arrays are out of sync! X={len(X)}, Y={len(Y_df)}")

    return X, Y_df
# --- MAIN LOOP ---

print(f"Starting pipeline for Subject: {subject_id}")
for session_id in all_sessions:
    print(f"\n--- Processing Session {session_id} ---")

    try:
        # 1. Load Data
        bids_path = mne_bids.BIDSPath(
            subject=subject_id, session=session_id, datatype='eeg',
            task=task, root=bids_root
        )
        print(f"looking for path :: {bids_path}")

        raw = mne_bids.read_raw_bids(bids_path=bids_path, verbose=True, extra_params={'preload': False}, on_ch_mismatch='reorder')
        print(f"    Loaded raw data: {raw.info['nchan']} channels, {raw.info['sfreq']} Hz.")
        print(f"    Channels found in raw data after loading: {raw.ch_names}") # <-- Added this line for debugging

        # 2. Preprocess (Using your provided function)
        raw = preprocess_raw(raw)

        # 3. Epoch and Label
        X_session, Y_session = label_and_epoch_data(raw, session_id)

        # 4. Store Results
        all_X_data.append(X_session)
        all_Y_data.append(Y_session)

        print(f"    Session {session_id} successful. Extracted {len(X_session)} epochs.")

    except FileNotFoundError:
        print(f"❌ File not found for Session {session_id}. Skipping.")
    except Exception as e:
        print(f"❌ An error occurred in Session {session_id}: {e}")

# --- 1. FINAL AGGREGATION ---

if all_X_data:
    X_combined = np.concatenate(all_X_data, axis=0)
    Y_combined = pd.concat(all_Y_data, ignore_index=True)

    print("\n==============================================")
    print("✅ Pipeline Complete.")
    print(f"Total Combined Trials (X): {X_combined.shape[0]}")
    print(f"Final Data Shape: (Trials, Channels, Timepoints) = {X_combined.shape}")
else:
    print("\nPipeline finished, but no data was processed.")
    X_combined = None
    Y_combined = None

# --- 2. CLASSIFIER PREPARATION (SCALING) ---

if X_combined is not None:

    # 2.1. Define Target Label
    target_label = 'is_recalled'  # <--- CORRECTED
    y = Y_combined[target_label].values

    # 2.2. Split into Training and Testing sets
    X_train, X_test, y_train, y_test = train_test_split(
        X_combined,
        y,
        test_size=0.2,
        stratify=y, # Still stratify, as recall will also be imbalanced
        random_state=42
    )

    # 2.3. Apply RobustScaler
    scaler = RobustScaler()

    n_trials_train, n_chans, n_times = X_train.shape
    X_train_reshaped = X_train.reshape(n_trials_train * n_chans, n_times)

    print("\nFitting RobustScaler on training data...")
    scaler.fit(X_train_reshaped)

    # Transform training data
    X_train_scaled_reshaped = scaler.transform(X_train_reshaped)
    X_train_scaled = X_train_scaled_reshaped.reshape(n_trials_train, n_chans, n_times)

    # Transform test data
    n_trials_test = X_test.shape[0]
    X_test_reshaped = X_test.reshape(n_trials_test * n_chans, n_times)
    X_test_scaled_reshaped = scaler.transform(X_test_reshaped)
    X_test_scaled = X_test_scaled_reshaped.reshape(n_trials_test, n_chans, n_times)

    print("✅ Data standardized and ready for classification (Task: Recall SME).")
    print(f"X_train_scaled shape: {X_train_scaled.shape}")
    print(f"X_test_scaled shape: {X_test_scaled.shape}")


'''
